In [33]:
import json
from pathlib import Path
import pandas as pd
import re
from typing import Any, Dict, List, Tuple

def parse_filename(filename):
    """Parse filename to extract registration number and date"""
    name = filename.replace('.json', '')
    match = re.match(r'^(\d+)_(\d{4}-?\d{2}-?\d{2})$', name)
    
    if match:
        reg_number = match.group(1)
        date_str = match.group(2)
        normalized_date = date_str.replace('-', '')
        return (reg_number, normalized_date)
    
    return (None, None)

def normalize_number(value):
    """Normalize number - treat 0/0.0/None as equivalent, convert to float for comparison"""
    if value is None or value == 0 or value == 0.0:
        return 0.0
    return float(value)

def normalize_list(value):
    """Normalize list for comparison - convert to sorted set"""
    if value is None:
        return set()
    if not isinstance(value, list):
        return {value}
    return set(value)

def compare_values(val1, val2, field_type='string'):
    """
    Compare two values based on field type
    Returns (is_match, val1_normalized, val2_normalized)
    """
    if field_type == 'number':
        norm1 = normalize_number(val1)
        norm2 = normalize_number(val2)
        return (norm1 == norm2, norm1, norm2)
    elif field_type == 'list':
        norm1 = normalize_list(val1)
        norm2 = normalize_list(val2)
        return (norm1 == norm2, sorted(norm1), sorted(norm2))
    else:  # string or other
        # Handle None values
        norm1 = str(val1) if val1 is not None else ''
        norm2 = str(val2) if val2 is not None else ''
        return (norm1 == norm2, norm1, norm2)

def compare_equipment_item(eq1, eq2, index, extracted_file, vix_file, discrepancies):
    """Compare a single equipment item and log discrepancies"""
    fields_to_compare = {
        'type': 'string',
        'numberOfUnits': 'number',
        'fuelType': 'string',
        'isEmergency': 'string',
        'electricalCapacity_kW': 'number',
        'mechanicalCapacity_bhp': 'number',
        'gasUsage_MMBTUhr': 'number'
    }
    
    for field, field_type in fields_to_compare.items():
        val1 = eq1.get(field)
        val2 = eq2.get(field)
        
        is_match, norm1, norm2 = compare_values(val1, val2, field_type)
        
        if not is_match:
            discrepancies.append({
                'extracted_file': extracted_file,
                'vix_file': vix_file,
                'section': 'equipmentSummary',
                'item_index': index,
                'field': field,
                'extracted_value': norm1,
                'vix_value': norm2,
                'mismatch_type': 'value_difference'
            })

def compare_operational_limit(op1, op2, index, extracted_file, vix_file, discrepancies):
    """Compare a single operational limit and log discrepancies"""
    # No fields to compare for operational limits
    pass

def compare_emission_limit(em1, em2, index, extracted_file, vix_file, discrepancies):
    """Compare a single emission limit and log discrepancies"""
    # No fields to compare for emission limits
    pass

def compare_matched_files():
    """
    Compare matched JSON files and log discrepancies
    """
    # Define paths
    extracted_data_path = Path('extracted_data')
    vix_outputs_path = Path('vix_outputs/results')
    
    # Check if paths exist
    if not extracted_data_path.exists():
        print(f"Error: {extracted_data_path} not found!")
        return None
    if not vix_outputs_path.exists():
        print(f"Error: {vix_outputs_path} not found!")
        return None
    
    # Collect files from extracted_data (year-wise folders)
    extracted_files = []
    extracted_file_map = {}  # filename -> full_path
    for year_dir in extracted_data_path.iterdir():
        if year_dir.is_dir():
            for f in year_dir.glob("*.json"):
                if not f.name.startswith('all_extracted_data_'):
                    reg_num, date = parse_filename(f.name)
                    if reg_num and date:
                        extracted_files.append((f.name, reg_num, date))
                        extracted_file_map[f.name] = f
    
    # Collect files from vix_outputs/results
    vix_files = []
    vix_file_map = {}  # filename -> full_path
    for f in vix_outputs_path.glob("*.json"):
        reg_num, date = parse_filename(f.name)
        if reg_num and date:
            vix_files.append((f.name, reg_num, date))
            vix_file_map[f.name] = f
    
    # Find matches
    matched_pairs = []
    for ext_name, ext_reg, ext_date in extracted_files:
        for vix_name, vix_reg, vix_date in vix_files:
            if ext_date == vix_date:
                if ext_reg.startswith(vix_reg) or vix_reg.startswith(ext_reg):
                    matched_pairs.append((ext_name, vix_name, ext_reg, vix_reg, ext_date))
                    break
    
    print(f"Found {len(matched_pairs)} matched pairs to compare")
    print("=" * 80)
    
    # Store all discrepancies
    discrepancies = []
    
    # Compare each matched pair
    for ext_name, vix_name, ext_reg, vix_reg, date in matched_pairs:
        print(f"Comparing: {ext_name} <-> {vix_name}")
        
        # Load JSON files
        try:
            with open(extracted_file_map[ext_name], 'r', encoding='utf-8') as f:
                extracted_data = json.load(f)
            with open(vix_file_map[vix_name], 'r', encoding='utf-8') as f:
                vix_data = json.load(f)
        except Exception as e:
            print(f"  Error loading files: {e}")
            discrepancies.append({
                'extracted_file': ext_name,
                'vix_file': vix_name,
                'section': 'file_loading',
                'item_index': None,
                'field': 'file_loading',
                'extracted_value': str(e),
                'vix_value': 'N/A',
                'mismatch_type': 'error'
            })
            continue
        
        # Compare top-level fields
        # Permit Issuance Date
        is_match, norm1, norm2 = compare_values(
            extracted_data.get('permitIssuanceDate'),
            vix_data.get('permitIssuanceDate'),
            'string'
        )
        if not is_match:
            discrepancies.append({
                'extracted_file': ext_name,
                'vix_file': vix_name,
                'section': 'top_level',
                'item_index': None,
                'field': 'permitIssuanceDate',
                'extracted_value': norm1,
                'vix_value': norm2,
                'mismatch_type': 'value_difference'
            })
        
        # Registration Number
        is_match, norm1, norm2 = compare_values(
            extracted_data.get('registrationNumber'),
            vix_data.get('registrationNumber'),
            'string'
        )
        if not is_match:
            discrepancies.append({
                'extracted_file': ext_name,
                'vix_file': vix_name,
                'section': 'top_level',
                'item_index': None,
                'field': 'registrationNumber',
                'extracted_value': norm1,
                'vix_value': norm2,
                'mismatch_type': 'value_difference'
            })
        
        # Compare equipmentSummary (removed array length check)
        extracted_equipment = extracted_data.get('equipmentSummary', [])
        vix_equipment = vix_data.get('equipmentSummary', [])
        
        # Compare each equipment item
        for i in range(min(len(extracted_equipment), len(vix_equipment))):
            compare_equipment_item(
                extracted_equipment[i],
                vix_equipment[i],
                i,
                ext_name,
                vix_name,
                discrepancies
            )
        
        # Compare operationalLimits (removed array length check)
        extracted_ops = extracted_data.get('operationalLimits', [])
        vix_ops = vix_data.get('operationalLimits', [])
        
        # Compare each operational limit
        for i in range(min(len(extracted_ops), len(vix_ops))):
            compare_operational_limit(
                extracted_ops[i],
                vix_ops[i],
                i,
                ext_name,
                vix_name,
                discrepancies
            )
        
        # Compare emissionLimits (removed array length check)
        extracted_emissions = extracted_data.get('emissionLimits', [])
        vix_emissions = vix_data.get('emissionLimits', [])
        
        # Compare each emission limit
        for i in range(min(len(extracted_emissions), len(vix_emissions))):
            compare_emission_limit(
                extracted_emissions[i],
                vix_emissions[i],
                i,
                ext_name,
                vix_name,
                discrepancies
            )
    
    # Create DataFrame from discrepancies
    if discrepancies:
        df = pd.DataFrame(discrepancies)
        
        # Save to CSV
        output_csv = 'json_comparison_discrepancies.csv'
        df.to_csv(output_csv, index=False)
        print(f"\n✓ Found {len(discrepancies)} discrepancies")
        print(f"✓ Saved to: {output_csv}")
        
        # Save to JSON for detailed inspection
        output_json = 'json_comparison_discrepancies.json'
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(discrepancies, f, indent=2)
        print(f"✓ Saved detailed JSON to: {output_json}")
        
        # Print summary statistics
        print("\n" + "=" * 80)
        print("DISCREPANCY SUMMARY:")
        print("=" * 80)
        print(f"Total discrepancies: {len(df)}")
        print(f"\nBy Section:")
        print(df['section'].value_counts().to_string())
        print(f"\nBy Mismatch Type:")
        print(df['mismatch_type'].value_counts().to_string())
        print(f"\nBy Field:")
        print(df['field'].value_counts().to_string())
        
        return df
    else:
        print("\n✓ No discrepancies found! All matched files are identical.")
        return pd.DataFrame()

# Run the comparison
discrepancies_df = compare_matched_files()

Found 150 matched pairs to compare
Comparing: 52299_2013-09-12.json <-> 52299_20130912.json
Comparing: 61610_2013-01-31.json <-> 61610_20130131.json
Comparing: 72296_2013-01-31.json <-> 72296_20130131.json
Comparing: 73870_2013-10-25.json <-> 73870_20131025.json
Comparing: 74020_2013-01-10.json <-> 74020_20130110.json
Comparing: 7237508_2014-04-04.json <-> 72375_20140404.json
Comparing: 73363_2014-07-16.json <-> 73363_20140716.json
Comparing: 73823_2014-11-17.json <-> 73823_20141117.json
Comparing: 73995_2014-05-08.json <-> 73995_20140508.json
Comparing: 74051_2014-08-08.json <-> 74051_20140808.json
Comparing: 74074_2022-05-18.json <-> 74074_20220518.json
Comparing: 74085_2022-11-21.json <-> 74085_20221121.json
Comparing: 73363_2014-07-16.json <-> 73363_20140716.json
Comparing: 73823_2014-11-17.json <-> 73823_20141117.json
Comparing: 73995_2014-05-08.json <-> 73995_20140508.json
Comparing: 74051_2014-08-08.json <-> 74051_20140808.json
Comparing: 74074_2022-05-18.json <-> 74074_20220518